# 06 — Individual Risk Modeling (raw microdata)

The rate sources (01–05) are **aggregate**. Here we use the **raw NCHS record-level microdata** (~11.9M births pooled over 2011–2013, one row per birth) to ask a different question:

> *What makes an individual birth more likely to end in infant death?*

We profile death rates by each birth-certificate factor, then fit **nested logistic-regression models** (birth factors → + maternal → + social), read the **odds ratios**, and check **discrimination (ROC-AUC)**.

**Note:** first run downloads ~500 MB and pools millions of rows, so it is slow. Descriptive rates use all births; deaths are double-counted by ~0.6% (they appear among the births too) — negligible here.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

from src.config import FIGURES_DIR, REPORTS_DIR
from src.data.sources import nber_microdata

sns.set_theme()
TABLES = REPORTS_DIR / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

df = nber_microdata.tidy()
print('births:', len(df), '| deaths:', int(df.died.sum()),
      '| IMR/1000:', round(df.died.mean()*1000, 2))

In [ ]:
# group the continuous covariates into interpretable categories
df['bw_cat'] = pd.cut(df.birthweight_g, [0,1500,2500,4000,9000],
                      labels=['<1500g','1500-2499g','2500-3999g','4000g+'])
df['gest_cat'] = pd.cut(df.gestation_wks, [0,28,32,37,50],
                        labels=['<28wk','28-31wk','32-36wk','37wk+'])
df['mage_cat'] = pd.cut(df.mother_age, [0,20,35,200], right=False, labels=['<20','20-34','35+'])
df['plural_cat'] = np.where(df.plurality == 1, 'Singleton', 'Multiple')
df['educ_cat'] = df.mother_educ.map({1:'<HS',2:'<HS',3:'HS',4:'Some college',
                                     5:'Some college',6:'College+',7:'College+',8:'College+'})
df['prenatal'] = np.where(df.prenatal_visits == 0, 'None', 'Some')
df['sex_cat'] = df.infant_sex

## Death rate by factor

Infant mortality per 1,000 within each level.

In [ ]:
def rate_table(col):
    g = df.groupby(col, observed=True)['died']
    return pd.DataFrame({'n': g.size(), 'deaths': g.sum(),
                         'imr_per_1000': (g.mean()*1000).round(1)})

for col in ['bw_cat','gest_cat','mother_race_hisp','educ_cat','prenatal','smoker']:
    print(col); print(rate_table(col), '\n')

In [ ]:
# save bar charts of IMR by the key factors
def bar(col, title, fname):
    t = rate_table(col)
    fig, ax = plt.subplots(figsize=(7,4))
    ax.bar(t.index.astype(str), t['imr_per_1000'], color='#1f3a5f')
    ax.set_ylabel('IMR per 1,000'); ax.set_title(title)
    plt.xticks(rotation=30, ha='right'); fig.tight_layout()
    fig.savefig(FIGURES_DIR / fname, dpi=130, bbox_inches='tight'); plt.show()

bar('bw_cat', 'IMR by birth weight', 'fig_imr_by_birthweight.png')
bar('gest_cat', 'IMR by gestational age', 'fig_imr_by_gestation.png')
bar('mother_race_hisp', 'IMR by maternal race/ethnicity', 'fig_imr_by_race.png')

## Nested logistic-regression models

Build risk in three layers and compare with likelihood-ratio tests and AIC:
1. **birth factors** — birth weight, gestation, plurality, infant sex
2. **+ maternal** — mother's age, smoking
3. **+ social** — education, race/ethnicity, prenatal care

Each categorical predictor is compared against a sensible reference level.

In [ ]:
feats = ['died','bw_cat','gest_cat','plural_cat','sex_cat','mage_cat',
         'smoker','educ_cat','mother_race_hisp','prenatal']
m = df[feats].dropna().copy()
print('complete-case rows:', len(m))

R = {'bw': "C(bw_cat, Treatment('2500-3999g'))", 'gest': "C(gest_cat, Treatment('37wk+'))",
     'plural': "C(plural_cat, Treatment('Singleton'))", 'sex': "C(sex_cat, Treatment('F'))",
     'mage': "C(mage_cat, Treatment('20-34'))", 'educ': "C(educ_cat, Treatment('College+'))",
     'race': "C(mother_race_hisp, Treatment('White (NH)'))", 'prenatal': "C(prenatal, Treatment('Some'))"}
f1 = f"died ~ {R['bw']} + {R['gest']} + {R['plural']} + {R['sex']}"
f2 = f1 + f" + {R['mage']} + smoker"
f3 = f2 + f" + {R['educ']} + {R['race']} + {R['prenatal']}"
m1 = smf.logit(f1, data=m).fit(disp=0)
m2 = smf.logit(f2, data=m).fit(disp=0)
m3 = smf.logit(f3, data=m).fit(disp=0)

In [ ]:
def lr(small, big):
    stat = 2*(big.llf - small.llf); ddf = int(big.df_model - small.df_model)
    return round(stat,1), ddf, stats.chi2.sf(stat, ddf)

comp = pd.DataFrame({'pseudo_r2': [round(x.prsquared,4) for x in (m1,m2,m3)],
                     'aic': [round(x.aic,1) for x in (m1,m2,m3)]},
                    index=['1: birth','2: +maternal','3: +social'])
comp.to_csv(TABLES / 'tbl_model_comparison.csv')
print(comp)
print('LR 1->2:', lr(m1,m2)); print('LR 2->3:', lr(m2,m3))

In [ ]:
# odds ratios from the full model, with 95% CIs
ci = np.exp(m3.conf_int())
odds = pd.DataFrame({'odds_ratio': np.exp(m3.params), 'ci_low': ci[0],
                     'ci_high': ci[1], 'p_value': m3.pvalues}).round(4)
odds.to_csv(TABLES / 'tbl_odds_ratios.csv')
odds

In [ ]:
# how well does the model rank a random death above a random survivor? (ROC-AUC)
tr, te = train_test_split(m, test_size=0.3, random_state=42, stratify=m.died)
fit_tr = smf.logit(f3, data=tr).fit(disp=0)
prob = fit_tr.predict(te)
auc = roc_auc_score(te.died, prob)
fpr, tpr, _ = roc_curve(te.died, prob)
fig, ax = plt.subplots(figsize=(6,6))
ax.plot(fpr, tpr, color='#1f3a5f', label=f'AUC = {auc:.3f}'); ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlabel('false positive rate'); ax.set_ylabel('true positive rate')
ax.set_title('ROC — infant-death risk model'); ax.legend()
fig.savefig(FIGURES_DIR / 'fig_roc_micro.png', dpi=130, bbox_inches='tight'); plt.show()
print('test-set ROC-AUC =', round(auc, 4))

## Notes

- **Birth weight and gestational age dominate.** Risk is orders of magnitude higher for very-low-birth-weight and extremely preterm infants; the model discriminates very well on these alone (high AUC).
- **Social factors add little** once birth factors are in: the AIC/pseudo-R² barely move from model 2 to 3, even though race and education are 'significant' on their own. A clear case of statistical vs practical significance, and of mediation (social factors act *through* birth weight/prematurity).
- Birth weight and gestation are highly collinear — read their coefficients together.
- Observational data: associations, not causal effects. The ~0.6% death double-count slightly biases the intercept, not the gradients.